# 📖 Notebook 3: Exactly-Once Semantics

When you send a message through Kafka, how many times does the receiver actually get it? Once? Twice? Maybe not at all?

This is the **delivery guarantee** problem — one of the most important concepts in distributed systems.

## 🎯 What You'll Learn

By the end of this notebook, you'll understand:

1. **The three delivery guarantees**: at-most-once, at-least-once, exactly-once
2. **Why exactly-once is hard** — and why most systems don't bother
3. **How idempotent producers prevent duplicates** with a single config flag
4. **How transactional messaging works** — atomic all-or-nothing writes
5. **Consumer offset management strategies** — and when to use each one

## Prerequisites

- Completed Notebook 1 (Producer/Consumer basics) and Notebook 2 (Consumer Groups)
- Kafka running locally via Docker Compose
- Basic Python knowledge

## 🛠️ Setup

### 1. Start Kafka

Open a terminal and run:

```bash
cd 03-technologies/messaging/kafka
docker-compose up -d
```

### 2. Select the correct kernel

In VS Code, click the **kernel picker** (top-right of this notebook) and select the **`.venv`** Python environment.

If the `.venv` kernel doesn't appear:
- Press `Cmd+Shift+P` → type **"Reload Window"** → hit Enter
- Then try selecting the kernel again

### 3. Run the setup cell below to verify your connection

In [ ]:
from confluent_kafka import Producer, Consumer
from confluent_kafka.admin import AdminClient, NewTopic
import json, time

KAFKA_CONFIG = {'bootstrap.servers': 'localhost:9092'}

admin = AdminClient(KAFKA_CONFIG)
try:
    metadata = admin.list_topics(timeout=5)
    print("✅ Connected to Kafka!")
except Exception as e:
    print(f"❌ Cannot connect: {e}")
    print("   Run: cd 03-technologies/messaging/kafka && docker-compose up -d")

## 📦 The Three Delivery Guarantees

Imagine you're sending a package to a friend. There are three different shipping services you can choose from:

### 1. At-Most-Once — "Fire and Forget" 🔥

**Analogy:** Like dropping a letter in a mailbox with no tracking number. It's fast and easy, but if the letter gets lost along the way, you'll never know. You sent it once and moved on.

- The producer sends the message and **doesn't wait for confirmation**
- If something goes wrong (network issue, broker crash), the message is **lost forever**
- **You get the message zero or one time** — never more than once

### 2. At-Least-Once — "Tracked Package" 📬

**Analogy:** Like sending a tracked package — if delivery confirmation doesn't arrive, you resend it. Your friend will definitely get the package, but they might accidentally get two copies if the tracking system glitches.

- The producer **waits for confirmation** from the broker
- If confirmation doesn't arrive, it **retries** sending
- You **never lose** a message, but you **might get duplicates**
- This is **Kafka's default** behavior!

### 3. Exactly-Once — "Bank Transfer" 🏦

**Analogy:** Like a bank transfer — the money moves from your account to your friend's account **exactly once**. The bank uses complex systems (transaction logs, two-phase commits) to make absolutely sure the transfer isn't lost AND isn't duplicated.

- The message is delivered **exactly one time** — no loss, no duplicates
- Requires **extra coordination** between producer, broker, and consumer
- **Hardest to achieve** and **slowest**, but critical for financial systems

### Comparison Table

| Guarantee | Speed | Data Loss? | Duplicates? | Use Case |
|-----------|-------|-----------|-------------|----------|
| At-most-once | ⚡ Fastest | Possible | No | Metrics, logging |
| At-least-once | 🚀 Fast | No | Possible | Most applications |
| Exactly-once | 🐢 Slowest | No | No | Payments, inventory |

> **Key insight:** There's always a trade-off between speed and safety. Choose based on what your application actually needs!

## 1️⃣ At-Most-Once Delivery

This is the simplest (and fastest) mode. The producer sends the message and immediately moves on without waiting for any confirmation from the broker.

**How it works:**
- `acks=0` — The producer doesn't wait for the broker to say "I got it"
- `retries=0` — If sending fails, don't try again
- The message might be lost, but we **never** send duplicates

**When to use:** When speed matters more than completeness — like collecting website analytics or system metrics. Losing a few data points is fine.

In [ ]:
topic = 'at-most-once-demo'
t = NewTopic(topic, num_partitions=1, replication_factor=1)
fs = admin.create_topics([t])
for n, f in fs.items():
    try:
        f.result()
    except:
        pass

# At-most-once: no retries, no waiting for acks
fire_and_forget_config = {
    **KAFKA_CONFIG,
    'acks': '0',           # Don't wait for broker acknowledgment
    'retries': 0,          # No retries
    'linger.ms': 0,        # Send immediately
}

producer = Producer(fire_and_forget_config)

print("🚀 At-Most-Once: Fire and Forget")
print("=" * 40)
for i in range(5):
    producer.produce(topic, value=f"metric-{i}".encode())
    print(f"  Sent metric-{i} (no confirmation)")

producer.flush()
print("\n⚠️ Messages sent but we have NO guarantee they arrived!")
print("💡 This is the fastest mode — good for metrics where losing some data is OK")

## 2️⃣ At-Least-Once Delivery (Kafka's Default)

This is how Kafka works out of the box, and it's the right choice for most applications.

**How it works on the producer side:**
- `acks=all` — Wait for **all** replicas to confirm they stored the message
- `retries=5` — If sending fails, try again up to 5 times
- We get a **delivery callback** confirming each message was stored

**How it works on the consumer side:**
- `enable.auto.commit=False` — We manually control when to mark messages as "processed"
- We commit the offset **after** processing each message
- If the consumer crashes after processing but **before** committing, the message will be delivered again on restart

**The catch:** If the consumer crashes between processing and committing its offset, the message gets reprocessed. That's why it's called "at-least-once" — you might process the same message more than once.

In [ ]:
topic2 = 'at-least-once-demo'
t = NewTopic(topic2, num_partitions=1, replication_factor=1)
fs = admin.create_topics([t])
for n, f in fs.items():
    try:
        f.result()
    except:
        pass

# Producer: reliable delivery with retries
reliable_config = {
    **KAFKA_CONFIG,
    'acks': 'all',          # Wait for all replicas to acknowledge
    'retries': 5,           # Retry up to 5 times
    'retry.backoff.ms': 100,
}

producer = Producer(reliable_config)

print("📨 At-Least-Once: Reliable Delivery")
print("=" * 40)

delivered = []
def on_delivery(err, msg):
    if err:
        print(f"  ❌ Failed: {err}")
    else:
        delivered.append(msg.offset())
        print(f"  ✅ Confirmed at offset {msg.offset()}")

for i in range(5):
    value = json.dumps({"payment_id": f"PAY-{i}", "amount": (i+1) * 10.0}).encode()
    producer.produce(topic2, value=value, callback=on_delivery)

producer.flush()
print(f"\n✅ All {len(delivered)} messages confirmed by broker!")

time.sleep(1)

# Consumer: manual commit AFTER processing
consumer_config = {
    **KAFKA_CONFIG,
    'group.id': 'at-least-once-demo',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,  # We control when to commit!
}

consumer = Consumer(consumer_config)
consumer.subscribe([topic2])

print("\n📥 Consumer: Manual offset commit (at-least-once)")
print("=" * 40)

empty = 0
while empty < 3:
    msg = consumer.poll(2.0)
    if msg is None:
        empty += 1
        continue
    if msg.error():
        continue
    empty = 0
    
    # Step 1: Process the message
    payment = json.loads(msg.value().decode())
    print(f"  Processing: {payment}")
    
    # Step 2: Commit AFTER processing
    consumer.commit(message=msg)
    print(f"    → Offset {msg.offset()} committed ✓")

consumer.close()
print("\n💡 If we crash between process and commit, the message gets reprocessed")
print("   That's 'at-least-once' — never lost, but possible duplicates")

## ⚠️ The Duplicate Problem

Let's see why at-least-once can be dangerous for certain applications.

**Imagine this scenario with a payment processor:**

```
1. Consumer reads message: "Charge Alice $100"     ✅
2. Consumer processes it: charges Alice's card       ✅
3. Consumer is about to commit the offset...         
   💥 CRASH! The consumer dies before committing.
4. Consumer restarts, reads from last committed offset
5. Consumer reads the SAME message: "Charge Alice $100"  😱
6. Consumer processes it AGAIN: charges Alice $100 AGAIN! 💸💸
```

**Alice just got charged $200 instead of $100!**

This is why we need exactly-once semantics for critical operations like:
- 💳 Payment processing
- 📦 Inventory management
- 🏦 Bank transfers
- 📊 Financial reporting

Kafka provides two mechanisms to solve this:
1. **Idempotent producers** — prevent duplicate writes to a topic
2. **Transactions** — atomic read-process-write cycles

## 🔒 Idempotent Producers

The word **"idempotent"** means: doing something multiple times has the same effect as doing it once.

**The problem idempotent producers solve:**

```
1. Producer sends message "Transfer $100"            →  Broker receives it ✅
2. Broker sends acknowledgment back to producer       →  Network glitch! Lost! ❌
3. Producer thinks it failed, retries the same message
4. Broker receives "Transfer $100" AGAIN              →  Duplicate! 😱
```

**How idempotent producers fix this:**

When you set `enable.idempotence=True`, the producer assigns a **sequence number** to every message. The broker keeps track of these numbers. If it receives the same sequence number twice, it simply discards the duplicate.

```
1. Producer sends message (seq=42) "Transfer $100"   →  Broker stores it ✅
2. Acknowledgment lost in network                     →  ❌
3. Producer retries message (seq=42) "Transfer $100"  →  Broker says: "Already got seq=42, ignoring" ✅
```

**The best part?** You only need to change ONE config setting — no code changes!

In [ ]:
topic3 = 'idempotent-demo'
t = NewTopic(topic3, num_partitions=1, replication_factor=1)
fs = admin.create_topics([t])
for n, f in fs.items():
    try:
        f.result()
    except:
        pass

idempotent_config = {
    **KAFKA_CONFIG,
    'enable.idempotence': True,  # The magic flag!
    'acks': 'all',               # Required for idempotency
}

producer = Producer(idempotent_config)

print("🔒 Idempotent Producer")
print("=" * 40)
print("Each message gets a unique sequence number.")
print("If a retry sends the same message twice, the broker ignores the duplicate.\n")

confirmed = []
def on_delivery(err, msg):
    if err:
        print(f"  ❌ {err}")
    else:
        confirmed.append(msg.offset())
        print(f"  ✅ Message at offset {msg.offset()} (broker will reject duplicates of this)")

for i in range(5):
    producer.produce(topic3, value=f"transfer-{i}".encode(), callback=on_delivery)

producer.flush()
print(f"\n✅ {len(confirmed)} messages sent with idempotency enabled")
print("💡 Even if the network caused retries, no duplicates in the topic!")

## 🔄 Transactional Messaging

Idempotent producers prevent duplicates from **retries**, but what about more complex scenarios?

**Consider a common Kafka pattern: read → process → write**

```
1. Read a message from Topic A         (e.g., "New order placed")
2. Process it                          (e.g., calculate total, validate)
3. Write result to Topic B             (e.g., "Order confirmed")
4. Commit the consumer offset for Topic A
```

What if we crash between step 3 and step 4? We wrote the result to Topic B, but didn't commit the offset. On restart, we'll read the same message from Topic A and write another result to Topic B — **duplicate output!**

**Kafka Transactions solve this** by making steps 3 and 4 **atomic** — they either BOTH happen or NEITHER happens.

### How Transactions Work

```
begin_transaction()
    ├── produce() to output topic          # Write result
    ├── send_offsets_to_transaction()       # Commit consumer offset
    └── commit_transaction()               # Make everything visible at once
```

If anything fails, call `abort_transaction()` — everything rolls back as if nothing happened.

**Think of it like a database transaction:** either ALL changes commit, or NONE of them do.

In [ ]:
input_topic = 'transfers-in'
output_topic = 'transfers-out'

for t_name in [input_topic, output_topic]:
    t = NewTopic(t_name, num_partitions=1, replication_factor=1)
    fs = admin.create_topics([t])
    for n, f in fs.items():
        try:
            f.result()
        except:
            pass

# Seed input topic
seed_producer = Producer(KAFKA_CONFIG)
for i in range(3):
    data = json.dumps({"id": f"TXN-{i}", "from": "Alice", "to": "Bob", "amount": (i+1)*100}).encode()
    seed_producer.produce(input_topic, value=data)
seed_producer.flush()
print("✅ Seeded 3 transfers into 'transfers-in'\n")

time.sleep(1)

# Transactional consumer-producer pattern
txn_producer_config = {
    **KAFKA_CONFIG,
    'transactional.id': 'transfer-processor-1',
    'enable.idempotence': True,
    'acks': 'all',
}

txn_producer = Producer(txn_producer_config)
txn_producer.init_transactions()

consumer_config = {
    **KAFKA_CONFIG,
    'group.id': 'transfer-processors',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,
}
consumer = Consumer(consumer_config)
consumer.subscribe([input_topic])

print("🔄 Transactional Processing: Read → Process → Write (atomically)")
print("=" * 60)

empty = 0
while empty < 3:
    msg = consumer.poll(2.0)
    if msg is None:
        empty += 1
        continue
    if msg.error():
        continue
    empty = 0
    
    transfer = json.loads(msg.value().decode())
    print(f"\n  📥 Read: {transfer}")
    
    # Process: add a processing timestamp and status
    result = {**transfer, "status": "completed", "processed_at": time.time()}
    
    # Atomic transaction: produce result + commit consumer offset
    txn_producer.begin_transaction()
    try:
        txn_producer.produce(output_topic, value=json.dumps(result).encode())
        
        # Commit consumer offset as part of the transaction
        txn_producer.send_offsets_to_transaction(
            consumer.position(consumer.assignment()),
            consumer.consumer_group_metadata(),
        )
        
        txn_producer.commit_transaction()
        print(f"  ✅ Transaction committed: {transfer['id']} → completed")
    except Exception as e:
        txn_producer.abort_transaction()
        print(f"  ❌ Transaction aborted: {e}")

consumer.close()
print("\n🎉 All transfers processed exactly once!")
print("💡 If anything fails mid-transaction, EVERYTHING rolls back — no partial writes!")

## 📍 Consumer Offset Strategies

How and when you commit offsets determines your delivery guarantee on the consumer side. Let's compare the three approaches:

### Strategy 1: Auto-Commit (Simplest)

```python
consumer_config = {
    'enable.auto.commit': True,      # Default!
    'auto.commit.interval.ms': 5000, # Commit every 5 seconds
}
```

**How it works:** Kafka automatically commits offsets in the background every 5 seconds.

**Risk:** If the consumer commits an offset but crashes **before** processing the message, that message is **lost** (at-most-once). If it processes the message but crashes **before** the next auto-commit, the message gets **reprocessed** (at-least-once).

**Best for:** Simple consumers where occasional duplicates or losses are acceptable.

---

### Strategy 2: Manual Commit After Processing (Recommended Default)

```python
consumer_config = {
    'enable.auto.commit': False,  # We control commits
}

# In your processing loop:
message = consumer.poll()
process(message)                    # Step 1: process first
consumer.commit(message=message)    # Step 2: then commit
```

**How it works:** You explicitly commit the offset **after** successfully processing each message (or batch).

**Guarantee:** At-least-once. If you crash between processing and committing, the message gets reprocessed on restart — but never lost.

**Best for:** Most applications. Combine with idempotent processing for best results.

---

### Strategy 3: Transactional Commit (Strongest)

```python
txn_producer.begin_transaction()
txn_producer.produce(output_topic, value=result)
txn_producer.send_offsets_to_transaction(offsets, group_metadata)
txn_producer.commit_transaction()
```

**How it works:** The consumer offset is committed **atomically** with the output message — as part of one transaction.

**Guarantee:** Exactly-once. Either both the output AND the offset commit happen, or neither does.

**Best for:** Financial transactions, inventory systems, or any read-process-write pipeline where duplicates are unacceptable.

---

### Quick Comparison

| Strategy | Config | Guarantee | Complexity |
|----------|--------|-----------|------------|
| Auto-commit | `enable.auto.commit=True` | At-most-once / At-least-once | Simple |
| Manual commit | `enable.auto.commit=False` | At-least-once | Moderate |
| Transactional | `transactional.id` + `send_offsets_to_transaction` | Exactly-once | Complex |

## 🧭 Choosing the Right Guarantee

Not every application needs exactly-once semantics. Here's a practical decision guide:

### When to use At-Most-Once
- 📊 **Logging and metrics** — Losing a few log lines is fine
- 📡 **Sensor data with high frequency** — Missing one reading out of thousands doesn't matter
- 🔔 **Non-critical notifications** — If a notification is lost, it's not the end of the world

### When to use At-Least-Once (Most Common)
- 📧 **Email/notification systems** — Better to send twice than not at all
- 🔍 **Search index updates** — Re-indexing a document is safe
- 📝 **Event logging** — Duplicate events can be filtered later
- ✅ **Any system with idempotent consumers** — If processing a message twice gives the same result, duplicates don't matter!

### When to use Exactly-Once
- 💳 **Payment processing** — Can't charge a customer twice
- 📦 **Inventory management** — Can't subtract stock twice for one order
- 🏦 **Financial transfers** — Money must move exactly once
- 📊 **Accurate aggregations** — Counting each event exactly once matters

### 💡 Pro Tip: Idempotent Consumers

Often, the simplest approach is to use **at-least-once delivery** and make your **consumer logic idempotent**:

```python
# Instead of:
balance -= transfer_amount  # Dangerous if processed twice!

# Do this:
if transfer_id not in processed_transfers:  # Check if already processed
    balance -= transfer_amount
    processed_transfers.add(transfer_id)
```

By tracking which messages you've already processed (using a database or cache), you can safely handle duplicates without the complexity of Kafka transactions.

## 📝 Key Takeaways

Let's recap everything we learned:

- **Three delivery guarantees exist:** at-most-once, at-least-once, and exactly-once — each with different trade-offs between speed and safety
- **Kafka defaults to at-least-once** — messages are never lost, but duplicates are possible. This is the safe, simple default for most apps
- **Idempotent producers** (`enable.idempotence=True`) prevent duplicates caused by network retries — just flip a config flag!
- **Transactions** provide full exactly-once semantics: atomic produce + offset commit, all-or-nothing
- **Making your consumer idempotent** (same input → same result, safe to retry) is often a simpler alternative to Kafka transactions
- **Choose based on your use case** — not everything needs exactly-once. Logging? At-most-once is fine. Payments? You need exactly-once

## ⏭️ What's Next

In the **next notebook**, we'll explore **real-time stream processing patterns** with Kafka — how to build pipelines that transform, filter, and aggregate data as it flows through your system.

---

**Happy learning! 🎓**